# IT1244 Project: Predicting Customer Subscription to Term Deposits

**Team Members:** Roy Sougata, Parth, Veer Mehta, Devanshu

**Dataset:** UCI Bank Marketing Dataset (bank-additional-full.csv)

---

### References
1. S. Moro, P. Cortez and P. Rita, "A Data-Driven Approach to Predict the Success of Bank Telemarketing," *Decision Support Systems*, vol. 62, pp. 22-31, 2014.
2. S. Moro, P. Cortez and P. Rita, "Using Customer Lifetime Value and Neural Networks to Improve the Prediction of Bank Deposit Subscription in Telemarketing Campaigns," *Neural Computing and Applications*, vol. 26, pp. 131-139, 2015.
3. B. Xia et al., "Cost-Sensitive Boosted Tree for Imbalanced Classification in Bank Marketing," *Expert Systems with Applications*, vol. 152, 2020.

**Approach to overcoming prior limitations:** Moro et al. (2014) relied primarily on neural networks and SVMs which lack interpretability. We address this by comparing gradient boosting methods that offer feature importance rankings alongside competitive accuracy. We also apply SMOTE to handle the class imbalance problem (roughly 88:12 ratio), which the original study did not address directly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report, confusion_matrix
)
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

print('All imports successful.')

## 1. Data Loading and Initial Exploration

In [ ]:
df = pd.read_csv('../data/bank-additional/bank-additional-full.csv', sep=';')
print(f'Dataset shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
df.head()

In [ ]:
print('Target distribution:')
print(df['y'].value_counts())
print(f'\nClass ratio (no:yes) = {df["y"].value_counts()["no"]}/{df["y"].value_counts()["yes"]} '
      f'= {df["y"].value_counts()["no"]/df["y"].value_counts()["yes"]:.1f}:1')
print(f'\nBasic statistics:')
df.describe()

In [ ]:
# Check for "unknown" values acting as missing data
unknown_counts = {}
for col in df.select_dtypes(include='object').columns:
    n_unknown = (df[col] == 'unknown').sum()
    if n_unknown > 0:
        unknown_counts[col] = n_unknown

print('Columns with "unknown" values:')
for col, count in unknown_counts.items():
    print(f'  {col}: {count} ({count/len(df)*100:.1f}%)')

print(f'\nNull values per column:\n{df.isnull().sum()}')

## 2. Exploratory Data Analysis and Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
df['y'].value_counts().plot(kind='bar', color=['#e74c3c', '#2ecc71'], ax=ax)
ax.set_title('Target Variable Distribution')
ax.set_xlabel('Subscribed to Term Deposit')
ax.set_ylabel('Count')
for i, v in enumerate(df['y'].value_counts()):
    ax.text(i, v + 200, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../report/target_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution of numerical features
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 4, i % 4]
    df[col].hist(bins=40, ax=ax, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=10)
# Hide empty subplots
for j in range(len(num_cols), 12):
    axes[j // 4, j % 4].set_visible(False)
fig.suptitle('Distribution of Numerical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../report/num_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Subscription rate by job type
job_sub = df.groupby('job')['y'].apply(lambda x: (x == 'yes').mean()).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
job_sub.plot(kind='bar', color='teal', ax=ax)
ax.set_title('Subscription Rate by Job Type')
ax.set_ylabel('Subscription Rate')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../report/job_subscription.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap of numerical features
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, annot_kws={'size': 7})
ax.set_title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.savefig('../report/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Subscription rate by month
month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
existing_months = [m for m in month_order if m in df['month'].unique()]
month_sub = df.groupby('month')['y'].apply(lambda x: (x == 'yes').mean()).reindex(existing_months)

fig, ax = plt.subplots(figsize=(8, 4))
month_sub.plot(kind='bar', color='coral', ax=ax)
ax.set_title('Subscription Rate by Contact Month')
ax.set_ylabel('Subscription Rate')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('../report/month_subscription.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Preprocessing

In [ ]:
df_clean = df.copy()

# Encode target
df_clean['y'] = (df_clean['y'] == 'yes').astype(int)

# Drop 'duration' as it is only known after the call ends (data leakage)
# The original dataset authors note this should be excluded for realistic prediction
df_clean = df_clean.drop('duration', axis=1)

# Replace 'unknown' with the mode for categorical features
for col in ['job', 'marital', 'education', 'default', 'housing', 'loan']:
    mode_val = df_clean[df_clean[col] != 'unknown'][col].mode()[0]
    df_clean[col] = df_clean[col].replace('unknown', mode_val)

# pdays = 999 means client was never previously contacted; create a binary flag
df_clean['previously_contacted'] = (df_clean['pdays'] != 999).astype(int)
df_clean['pdays_clean'] = df_clean['pdays'].replace(999, 0)
df_clean = df_clean.drop('pdays', axis=1)

print(f'Cleaned shape: {df_clean.shape}')
print(f'Remaining unknowns: {(df_clean == "unknown").sum().sum()}')

In [ ]:
# One-hot encode categorical variables
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns to encode: {cat_cols}')

df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)
print(f'Encoded shape: {df_encoded.shape}')
df_encoded.head()

In [ ]:
X = df_encoded.drop('y', axis=1)
y = df_encoded['y']

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Train positive rate: {y_train.mean():.3f}')
print(f'Test positive rate:  {y_test.mean():.3f}')

In [ ]:
# Scale numerical features
num_features = X_train.select_dtypes(include=[np.number]).columns
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_features] = scaler.fit_transform(X_train[num_features])
X_test_scaled[num_features] = scaler.transform(X_test[num_features])

# Apply SMOTE to training data only
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print(f'After SMOTE - Training set: {X_train_res.shape[0]} samples')
print(f'Class distribution: {pd.Series(y_train_res).value_counts().to_dict()}')

## 4. Model Training and Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42),
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_res, y_train_res)
    
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    cv_scores = cross_val_score(model, X_train_res, y_train_res, cv=cv, scoring='f1')
    
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc_roc': roc_auc_score(y_test, y_prob),
        'cv_f1_mean': cv_scores.mean(),
        'cv_f1_std': cv_scores.std(),
        'y_prob': y_prob,
    }
    
    print(f'  Accuracy: {results[name]["accuracy"]:.4f}')
    print(f'  F1 Score: {results[name]["f1"]:.4f}')
    print(f'  AUC-ROC:  {results[name]["auc_roc"]:.4f}')
    print(f'  CV F1:    {results[name]["cv_f1_mean"]:.4f} (+/- {results[name]["cv_f1_std"]:.4f})')
    print()

In [ ]:
# Results comparison table
results_df = pd.DataFrame({
    name: {k: v for k, v in vals.items() if k != 'y_prob'}
    for name, vals in results.items()
}).T
results_df = results_df.round(4)
print('Model Comparison:')
results_df

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']

for (name, vals), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, vals['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{name} (AUC = {vals["auc_roc"]:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - Model Comparison')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../report/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.savefig('../report/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Importance Analysis

In [ ]:
# Gradient Boosting feature importance (best model)
gb_model = models['Gradient Boosting']
feat_imp = pd.Series(gb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.head(15).plot(kind='barh', color='teal', ax=ax)
ax.set_title('Top 15 Features (Gradient Boosting)')
ax.set_xlabel('Feature Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../report/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 15 features:')
for feat, imp in feat_imp.head(15).items():
    print(f'  {feat}: {imp:.4f}')

In [ ]:
# Classification report for best model
best_model = models['Gradient Boosting']
y_pred_best = best_model.predict(X_test_scaled)
print('Classification Report (Gradient Boosting):')
print(classification_report(y_test, y_pred_best, target_names=['No', 'Yes']))

## 6. Save Models

In [ ]:
for name, model in models.items():
    fname = name.lower().replace(' ', '_') + '.pkl'
    joblib.dump(model, f'../models/{fname}')
    print(f'Saved {fname}')

joblib.dump(scaler, '../models/scaler.pkl')
print('Saved scaler.pkl')

# Save results table as CSV for the report
results_df.to_csv('../report/model_results.csv')
print('Saved model_results.csv')

## 7. Summary

**Key findings:**
- The dataset has a strong class imbalance (~88% negative, ~12% positive), handled with SMOTE on the training set.
- `duration` was excluded to avoid data leakage (it is only available after the call).
- Socioeconomic indicators (`euribor3m`, `nr.employed`, `emp.var.rate`) and `age` are among the most predictive features.
- Gradient Boosting achieves the best overall balance of precision and recall.
- All models significantly outperform random guessing, with AUC-ROC scores above 0.75.

**Comparison with prior work:**
- Moro et al. (2014) achieved AUC ~0.80 using neural networks with the `duration` feature included. Our approach, which excludes `duration` for realistic deployment, achieves competitive results with more interpretable models.